<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/18_2_0_Classification_LogReg_Intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to Logistic Regression with Titanic

Author: Brad Sheese


---

# **Introduction to Classification - Logistic Regression**

This notebook introduces 'classification' through logistic regression. Before you start into this you should be fairly comfortable with simple linear regression. If you're a bit fuzzy on simple linear regression go back and review simple linear regression exercises before you dig into this.


---
## **Goals of Classification with Logistic Regression**
When we use the word 'classify' we are typically taking some case or instance in our dataset and assigning it to some sort of discrete category. Let's say I had a large number of individual fruits and in my dataset I just had each piece of fruits physical dimensions (length, width, height). I also have labels for each fruit (apple or bannana). I want to know if the dimensions of the fruit can be used to predict the fruit category classification. Perhaps length is a good way to classify an apple vs. a bannana.

This is different than simple linear regression where I am usually predicting some kind of continuous or non-categorical outcome. This is also different from some of the cluster analysis approaches you may be familiar with where we were classifying things without knowing the label. Logistic regression is a form of supervised learning because we know the classification in advance. What we don't know is what parameters might help us make the classification. For this example, we want to discover if the dimensions of the fruit can help us differentiate the apples from the banannas.


---
## **Assumptions of Binary Logistic regression**
So let's get a bit more formal and specific. Our outcome of interest only has two categories (it's binary, apple or bannana). So the model we are going to develop is called a binary logistic regression. Binary logistic regression models assumes some things about your data. If our data don't fit these assumptions our model may be inaccurate.



### **Assumption 1: Binary Outcome**
Your data should have a category that can be represented by a 0 and 1. We might code all apples as 0 and bannans as 1.



### **Assumption 2: Low Levels of Multi-Colinearity**
The things you are using to predict the category (called parameters, or features, or independent variables) should not be strongly related to one another. This is formally called multi-colinearity. The stronger the association is between any two predictors, the bigger a problem this becomes.

Let's say I had the length of the fruit in metric and the length of the fruit in imperial as two seperate parameters in my data. They would have different means, but would be perfectly correlated. Using both of these as predictors in my model would violate the assumption of low or no multi-colinearity among the paramters and would make the result difficult to meaningfully interpret. In particular, the direction and magnitude of the parameter weights would not be accurate.  Consequently, you would remove one of them instead of using both.

You can test for violations of this assumption by examining 'tolerance' or the 'Variable Inflation Factor (VIF)'. If the correlations between predictor exceed .80 you definitely have a case for concern.



### **Assumption 3: Independent Errors**
Our model assumes that the cases are independent of one another and so are their errors. For example, the cases should not represent repetitions of the same thing over time. For out fruits, each case is a unique fruit, not the same fruit assessed repeatedly over time. More formally, errors should be independent or uncorrelated. If the errors are not independent, due to some kind of clustering or repeated assessments, the model will not work properly.



### **Assumption 4: Linear Relation Between Predictors and Logit of Outcome**
We will come back to this once we understand 'Logit of the Outcome' a bit better.



### **Assumption 5: Homoscedasticity**
Our model assumes that variance is constant across all values of a feature. [See wikipedia for an explanation](https://en.wikipedia.org/wiki/Homoscedasticity_and_heteroscedasticity).


---
### **Additional Considerations:**

* **Outliers:** As with any analysis, outliers may distort the results of your analysis. In any form regression, outliers may contribute to violations of the assumptions of the model and may have a disproportionate influence on parameters.
* **Sample size:** As with any analysis, large samples are better. Small sample sizes may contribute to violations of the assumptions of the model.






In [ ]:
%%capture
!pip install pingouin

In [ ]:
import pandas as pd
import seaborn as sns
from matplotlib import style
import matplotlib.pyplot as plt
import urllib.request as request
import numpy as np

import pingouin as pg

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import FunctionTransformer
from imblearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import train_test_split
from sklearn import linear_model
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import mean_squared_error, r2_score

# temp fix for scatter
from matplotlib.axes._axes import _log as matplotlib_axes_logger
matplotlib_axes_logger.setLevel('ERROR')

## **Logistic Regression Exercise: Predicting Surivial on the Titanic**
We've spent a lot of time with the Titanic data and by now you may have a pretty good sense of what factors are important in predicting survival. You would likely have gotten this sense from our previous work with descriptives statistics (comparing means and such) or with our visualizations. Logistic regression is going to be a step-up from that previous work in that we will be able to formally quantify the contribution of each predictor.

In [ ]:
url_root = 'https://raw.githubusercontent.com/'
url_repo = 'bsheese/CSDS125ExampleData/master/'
url_file = 'data_titanic.csv'

datafile = request.urlopen(url_root + url_repo + url_file)
df = pd.read_csv(datafile)
df.columns = df.columns.str.lower()
df.columns = [x.replace(' aboard', '') for x in df.columns]
df.info()

## **Cleaning and Preparing the Dataset**

### **Check for Missing Values**
This version of the titanic data has no missing values, but its always good to check for missing values and address them before you start into the main analysis.

In [ ]:
df.isna().sum()

### **Identifying Categorical and Non-Categorical Variables**


In [ ]:
#
df.nunique()

In [ ]:
#

for s in ['siblings/spouses', 'parents/children']:
  print(s)
  print(df[s].value_counts())
  sns.histplot(df[s])
  plt.show()
  print()


In [ ]:
#
df.loc[df['siblings/spouses'] == 0, 'sibspouse'] = 0
df.loc[df['siblings/spouses'] != 0, 'sibspouse'] = 1
df.loc[df['parents/children'] == 0, 'parentchild'] = 0
df.loc[df['parents/children'] != 0, 'parentchild'] = 1


In [ ]:
#
df.loc[:, 'sex'] = df.loc[:, 'sex'].replace({'male': 0, 'female': 1})

In [ ]:
#
cat_cols = ['survived', 'pclass', 'sex', 'sibspouse', 'parentchild']

for s in cat_cols:
  print(s)
  sns.countplot(x = df[s])
  plt.show()
  print()

In [ ]:
#
noncat_cols = ['age', 'fare']

print(df[noncat_cols].describe().T)

for s in noncat_cols:
  sns.histplot(x = df[s])
  plt.show()

In [ ]:
#
df.loc[df['fare'] > 99, 'fare_t'] = 100
df.loc[df['fare'] <= 99, 'fare_t'] = df['fare']

sns.histplot(df['fare_t'])
plt.show()

In [ ]:
#
df.loc[:, 'fare_tl'] = df.loc[:, 'fare_t'].map(np.log1p)
sns.histplot(df['fare_tl'])
plt.show()

In [ ]:
noncat_cols = ['age', 'fare_tl']

## **Checking Assumptions** - Just 1 and 2 for now

### **Assumption 1: Binary Outcome**

In [ ]:
df['survived'].value_counts()

### **Assumption 2: Low Levels of Multi-Colinearity**

In [ ]:
sns.pairplot(df[noncat_cols])
plt.show()

In [ ]:
sns.heatmap(df[noncat_cols].corr(), annot=True, cmap='Blues')
plt.show()

## **Dummy Coding (One Hot Encoding) Categorical Features**

In [ ]:
if not 'pclass_1' in df.columns:
  df = pd.get_dummies(df, columns = ['pclass'])
df.describe().T

## **Creating and Cross-Validating the Model**
We are going to validate our model by splitting our data into a training dataset and a testing dataset. We will train our model on the training dataset and then see how well is does in making predictions with the testing dataset.

This approach helps address concerns about 'overfitting' our data.


---
### Splitting the Data into Training and Test

`test_train_split` is a method from the scikit learn module designed to produce testing and training datasets. This method allows us to specify the amount of the dataset to set aside to use for testing using the `test_size` argument.

By convention `X` is used to represent the parameters (also called parameters, features, or independent variables).

By convention `y` is used to represent the labels (also called the outcome, or dependent variable).

In [ ]:
# defining features(X) and outcome(y)
X = df[['age', 'sibspouse','parentchild', 'fare_tl', 'sex', 'pclass_1', 'pclass_2', 'pclass_3']]
y = df['survived']

# using the train_test_split method to create testing and training datasets
trainX, testX, trainY, testY = train_test_split(X, y, test_size=0.25)

### **Creating and Training the Model**
We will use the LogisticRegression() function from the sci-kit learn linear regression module.

We need to choose a solver and specify the total number of iterations and then fit the model to the training (X-train, y_train).

Choosing a solver and setting the maximum iterations is beyond the scope of this introduction. If you do not specify these argument, the default solver and iterations will often do just fine. If you receive a warning that model has failed after reaching the maximum iterations you will need to dive deeper into this issue.

In [ ]:
# only scale a subset of columns
scaler = StandardScaler()
scale_cols = ['age', 'fare_tl']
scaler.fit(df[scale_cols])
df[scale_cols] = scaler.transform(df[scale_cols])


# create and fit model
model = LogisticRegression(solver='sag', max_iter=4000)

pipe = Pipeline(steps=[('regressor', model)])
pipe.fit(trainX.values, trainY)

print(f'Model Score, Train Set: {pipe.score(trainX.values, trainY.values).round(3)}')
print(f'Model Score, Test Set: {pipe.score(testX.values, testY.values).round(3)}')

In [ ]:
betas = pd.DataFrame(zip(trainX.columns, model.coef_[0]))
betas.columns = ['feature', 'beta']
betas.set_index('feature').sort_values(by='beta')

The coefficients in a logistic regression model are the log odds that the response is 1. In this specific case, the log odds of surviving the titanic. To convert this into something more familiar we 'exponentiate' the log odds to the odds ratio for a one unit increase in the parameter.

In [ ]:
betas['odds_ratio'] = betas['beta'].map(np.exp)
betas.sort_values('odds_ratio')

So how do you interpret the odds ratios? The values less than one indicate reduced odds of the outcome occurring while the values greater than one indicate increased odds of the outcome occurring.

In this case, the outcome is surviving (coded as 1 in our data) so sex turns out to be our best predictor. Females are coded as 1 and males are coded as 0. So our model indicates that females are 11  times as likely to survive.

  

### **Testing the Model with Cross-Validation**
Now that we have created a model with the training data we will apply the model to the test data and see how well it does.

In [ ]:
predictions = model.predict(testX.values)
print(classification_report(testY, predictions))

Pay attention to the weighted average under precision. Our model is about 80% accurate. Everytime you run this notebook you'll get a slightly different result because we are randomly selecting our testing and training data. However, whatever the exact value is, it will be close to 80%. That's not bad.

Let's look at the model's accuracy a slightly different way. We're going to use the `confusion_matrix` functions which takes our testing outcomes (`y_test` and our model's `predictions`) and shows us how are model is doing in practical terms.

In [ ]:
c_matrix = confusion_matrix(testY, predictions)
print('Our logistic regression model predicted the following when applied to the test data:')
print(f'It correctly predicted {c_matrix[0][0]} of {c_matrix[0][0] + c_matrix[0][1]} fatalities, but incorrectly predicted {c_matrix[0][1]} fatalities that did not occur.')
print(f'It correctly predicted {c_matrix[1][1]} of {c_matrix[1][0] + c_matrix[1][1]} survivors, but incorrectly predicted {c_matrix[1][0]} survivors that did not survive.')

# Problem 1:
Explain precision, recall, and f1 score. Be clear and precise about what each one measures.   

Write your answer here.

# Problem 2:
Look at the classification report for our model. Explain what the classification report values mean in terms of our model's performance.

Write your answer here.

# Problem 3
Write code to find the best model with only four features.

In [ ]:
# enter and test your code here


##Problem 4
Explain your solution to Problem 3. What did you find, how did you determine it is the best solution?

Write your answer here.